# 2 — Préparation des données (CRISP-DM Phase 3)

Ce notebook est la **source canonique** pour la préparation des données du projet Inved Corp. Il :

1. Charge les données (`./data/train.csv`).
2. Élimine les outliers et applique la transformation logarithmique sur la cible.
3. **Crée des variables dérivées** (âges, surfaces et bains agrégés, indicateurs binaires).
4. Sépare le train et le test (split déterministe) et retire les variables redondantes (multicolinéarité).
5. Construit **trois variantes de préprocesseur**, chacune justifiée selon les hypothèses des familles de modèles qui les consomment.
6. Définit des fonctions utilitaires partagées (`rmsle_score`, `predicted_vs_actual_plot`, `publish_result`).

Chaque notebook de modélisation (3a, 3b, 3c, 3d) démarre par :

```python
%load_ext autoreload
%autoreload 2
%run 2_data_prep.ipynb
```

…ce qui ré-exécute ce notebook dans son propre kernel et expose toutes les variables, transformateurs et fonctions utilitaires sans import ad-hoc.

## 3.1 Chargement des données et imports

In [1]:
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.base import BaseEstimator, TransformerMixin, RegressorMixin
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler, TargetEncoder

RANDOM_STATE = 42

In [2]:
df = pd.read_csv('./data/train.csv', sep=',')
print(f"Dimensions brutes : {df.shape}")

Dimensions brutes : (1460, 81)


## 3.2 Suppression des observations atypiques

Sur la base de l'analyse exploratoire (NB1) et du papier de référence **De Cock (2011)**, nous retirons uniquement les rares maisons dont la surface habitable (`GrLivArea > 4000`) **et** le prix de vente (`SalePrice < 300 000`) trahissent une vente forcée ou un défaut majeur — pas les maisons de luxe à très grande surface, qu'on veut garder pour que le modèle apprenne le régime haut-de-gamme.

In [3]:
outliers_index = df[(df['GrLivArea'] > 4000) & (df['SalePrice'] < 300_000)].index
print(f"Nombre de points supprimés : {len(outliers_index)}")

df_cleaned = df.drop(outliers_index).copy()
print(f"Dimensions après suppression : {df_cleaned.shape}")

Nombre de points supprimés : 2
Dimensions après suppression : (1458, 81)


## 3.3 Transformation logarithmique de la cible

La variable `SalePrice` est très asymétrique à droite. On applique `log1p` pour stabiliser la distribution. La métrique du projet est la **RMSLE** (RMSE sur `log1p(SalePrice)`) — une erreur de 10 k$ sur une maison à 100 k$ ne se mesure pas comme une erreur de 10 k$ sur une villa à 1 M$. La RMSLE pénalise *proportionnellement*, ce qui correspond au sens métier de Inved Corp.

In [4]:
df_cleaned['SalePrice_log'] = np.log1p(df_cleaned['SalePrice'])

## 3.4 Ingénierie de variables

Avant le découpage, on enrichit le jeu de données avec des variables dérivées calculées **ligne à ligne** — donc reproductibles à l'identique sur le test Kaggle (la fonction `engineer_features` est réutilisée dans le NB5). Chaque variable remplace avantageusement deux ou trois colonnes brutes corrélées :

| Variable dérivée    | Formule                                                     | Ce qu'elle capture |
|---------------------|-------------------------------------------------------------|--------------------|
| `HouseAge`          | `YrSold - YearBuilt`                                        | Âge du bien à la vente — signal plus direct que deux années absolues. |
| `YearsSinceRemodel` | `YrSold - YearRemodAdd`                                     | Effet « rénové récemment », distinct de « construit récemment ». |
| `GarageAge`         | `YrSold - GarageYrBlt` (borné à ≥ 0, `NaN` si pas de garage) | Âge du garage ; quelques lignes Ames ont `GarageYrBlt > YrSold` (erreur de saisie connue), on borne donc à 0. |
| `TotalSF`           | `1stFlrSF + 2ndFlrSF + TotalBsmtSF`                         | Surface totale — prédicteur de référence sur le concours Ames. |
| `TotalBathrooms`    | `FullBath + 0.5·HalfBath + BsmtFullBath + 0.5·BsmtHalfBath` | Nombre de salles de bain agrégé en une seule colonne. |
| `HasGarage`, `HasPool`, `HasSecondFloor` | `(surface correspondante > 0)`        | Indicateurs binaires — aident les arbres à séparer proprement les cas « sans ». |

On force également `MSSubClass` (code de type de logement : 20, 60, 75…) en **chaîne de caractères** : c'est une variable catégorielle déguisée en entier, qu'il ne faut surtout pas standardiser.

In [5]:
COLLINEAR_DROP_COLS = ['GarageArea', 'TotalBsmtSF', 'TotRmsAbvGrd', 'GarageYrBlt']


def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    """Variables dérivées, appliquées à l'identique au train et au test Kaggle.

    Transformation purement ligne-à-ligne (aucun ajustement, aucune fuite) :
    elle est rejouée telle quelle sur data/test.csv dans le NB5.
    """
    df = df.copy()

    # MSSubClass est un code catégoriel (20, 60, 75…), pas une grandeur numérique :
    # on le force en chaîne pour qu'il soit traité comme nominal, jamais mis à l'échelle.
    df['MSSubClass'] = df['MSSubClass'].astype(str)

    # Âges au moment de la vente — signal direct, vs deux années absolues corrélées.
    df['HouseAge'] = df['YrSold'] - df['YearBuilt']
    df['YearsSinceRemodel'] = df['YrSold'] - df['YearRemodAdd']
    df['GarageAge'] = (df['YrSold'] - df['GarageYrBlt']).clip(lower=0)  # NaN si pas de garage

    # Surfaces et bains agrégés — une colonne dérivée vaut mieux que 3-4 brutes.
    df['TotalSF'] = (df['1stFlrSF'].fillna(0)
                     + df['2ndFlrSF'].fillna(0)
                     + df['TotalBsmtSF'].fillna(0))
    df['TotalBathrooms'] = (df['FullBath'].fillna(0)
                            + 0.5 * df['HalfBath'].fillna(0)
                            + df['BsmtFullBath'].fillna(0)
                            + 0.5 * df['BsmtHalfBath'].fillna(0))

    # Indicateurs binaires — aident les arbres à isoler proprement les cas « sans ».
    df['HasGarage'] = (df['GarageArea'].fillna(0) > 0).astype(int)
    df['HasPool'] = (df['PoolArea'].fillna(0) > 0).astype(int)
    df['HasSecondFloor'] = (df['2ndFlrSF'].fillna(0) > 0).astype(int)

    return df


df_feat = engineer_features(df_cleaned)
print(f"Variables après ingénierie : {df_feat.shape[1]} colonnes "
      f"(+{df_feat.shape[1] - df_cleaned.shape[1]} dérivées)")

Variables après ingénierie : 90 colonnes (+8 dérivées)


In [6]:
new_feats = ['HouseAge', 'YearsSinceRemodel', 'GarageAge', 'TotalSF',
             'TotalBathrooms', 'HasGarage', 'HasPool', 'HasSecondFloor']

corr = (df_feat[new_feats + ['SalePrice_log']].corr()['SalePrice_log']
        .drop('SalePrice_log').sort_values(key=np.abs, ascending=False))

print("Corrélation des variables dérivées avec SalePrice_log :")
for name, r in corr.items():
    flag = "   <-- faible (|r| < 0.1)" if abs(r) < 0.1 else ""
    print(f"  {name:18s} : {r:+.3f}{flag}")

Corrélation des variables dérivées avec SalePrice_log :
  TotalSF            : +0.825
  TotalBathrooms     : +0.677
  HouseAge           : -0.588
  YearsSinceRemodel  : -0.569
  GarageAge          : -0.543
  HasGarage          : +0.323
  HasSecondFloor     : +0.151
  HasPool            : +0.077   <-- faible (|r| < 0.1)


## 3.5 Découpage train/test et gestion de la multicolinéarité

Le split est fait **avant** toute transformation (imputation, encodage, mise à l'échelle) afin de garantir qu'aucune statistique calculée sur le test ne fuite dans l'entraînement. Toutes les transformations seront ajustées **uniquement** sur `X_train` via les `Pipeline` / `ColumnTransformer` de la section 3.6.

Graine fixée à `42` pour la reproductibilité : chaque notebook de modélisation qui ré-exécute ce fichier obtiendra le même `X_train` / `X_test`.

**Multicolinéarité.** L'EDA (NB1) a mis en évidence plusieurs paires de variables quasi-redondantes ; on en retire une de chaque paire (`COLLINEAR_DROP_COLS`). Cela stabilise les coefficients des modèles linéaires (Ridge/Lasso/ElasticNet voient leur variance gonfler en présence de colinéarité) et rend les graphes d'importance honnêtes. On garde la variable la plus interprétable / la moins lacunaire :

| Conservée   | Retirée        | Raison |
|-------------|----------------|--------|
| `GarageCars`  | `GarageArea`   | `r > 0.88` ; le nombre de places est plus interprétable et légèrement plus prédictif sur la cible log. |
| `1stFlrSF`    | `TotalBsmtSF`  | `r ≈ 0.82` ; surface au sol, moins lacunaire (`TotalBsmtSF` est déjà résumé par `TotalSF`). |
| `GrLivArea`   | `TotRmsAbvGrd` | `r > 0.80` ; la surface habitable continue est plus informative que le nombre de pièces. |
| `YearBuilt`   | `GarageYrBlt`  | `r ≈ 0.83` et `GarageYrBlt` est lacunaire ; l'âge du garage est désormais porté par `GarageAge`. |

Les colonnes d'années / bains brutes restantes (`YearBuilt`, `FullBath`…) sont **conservées** à côté de leurs agrégats : les arbres en tirent encore un signal, et le Lasso saura annuler les redondances par régularisation.

In [7]:
X = df_feat.drop(columns=['SalePrice', 'SalePrice_log', 'Id'], errors='ignore')
y_log = df_feat['SalePrice_log']

X_train, X_test, y_train_log, y_test_log = train_test_split(
    X, y_log, test_size=0.2, random_state=RANDOM_STATE
)

# Retrait des paires colinéaires (cf. tableau ci-dessus) — appliqué après le split,
# une fois que les variables dérivées (TotalSF, GarageAge) ont consommé les colonnes brutes.
X_train = X_train.drop(columns=COLLINEAR_DROP_COLS, errors='ignore')
X_test = X_test.drop(columns=COLLINEAR_DROP_COLS, errors='ignore')

print(f"X_train : {X_train.shape}   |   X_test : {X_test.shape}")
print(f"Colonnes retirées (colinéarité) : {COLLINEAR_DROP_COLS}")
print(f"NaN dans X_train : {int(X_train.isna().sum().sum())} cellules sur {X_train.size}")

X_train : (1166, 83)   |   X_test : (292, 83)
Colonnes retirées (colinéarité) : ['GarageArea', 'TotalBsmtSF', 'TotRmsAbvGrd', 'GarageYrBlt']
NaN dans X_train : 6276 cellules sur 96778


## 3.6 Trois variantes de préprocesseur, une par famille de modèles

Chaque famille de modèles a des **hypothèses différentes sur les données d'entrée** ; il serait erroné d'appliquer le même prétraitement à tous. Nous construisons trois variantes nommées, et chaque notebook de modélisation choisira la sienne en fonction des hypothèses des modèles qu'il contient.

| Variante                  | Imputation             | Encodage catégoriel                                                          | Mise à l'échelle | Famille cible                                  |
|---------------------------|------------------------|------------------------------------------------------------------------------|------------------|-----------------------------------------------|
| `preprocessor_scaled`     | médiane (dont `LotFrontage` par quartier) / 0 / mode / `'None'` | Ordinal (qualités) + **TargetEncoder** (nominal haute card.) + OneHot (basse card.) | **StandardScaler** | Linéaires, KNN, SVR, MLP — sensibles à l'échelle |
| `preprocessor_encoded`    | idem                   | idem                                                                          | aucune           | Arbres sklearn (DT, RF, GB, AdaBoost)         |
| `preprocessor_native`     | **aucune** (NaN gérés par le modèle) | aucune (`pd.Categorical` brut)                                  | aucune           | XGBoost, LightGBM, CatBoost — gestion native  |

Les justifications détaillées (assumptions du modèle → choix du préprocesseur) sont rappelées dans chaque notebook de modélisation au début de chaque modèle (bloc « Hypothèses du modèle »).

### 3.6.1 Groupes de colonnes et audit de cardinalité

On définit les listes de colonnes à partir de `X_train`, en distinguant :

- **Numériques** où NA = `0` (absence d'équipement, ex. `BsmtFinSF1` quand pas de sous-sol) → imputation constante par `0`.
- **Numériques** où NA = vraie valeur manquante (ex. `LotFrontage`) → imputation par la médiane (avec, pour `LotFrontage`, une médiane **par quartier**, cf. 3.6.2).
- **Ordinales** : échelles de qualité (`Ex > Gd > TA > Fa > Po > None`) → `OrdinalEncoder` avec ordre explicite.
- **Nominales haute cardinalité** (≥ 6 modalités) → `TargetEncoder` (cf. justification ci-dessous).
- **Nominales basse cardinalité** (≤ 5 modalités) → `OneHotEncoder` (peu de colonnes, lisible).
- `Functional` reçoit une branche dédiée (imputation constante `'Typ'`, la valeur « normale » du dictionnaire, puis `TargetEncoder`).
- `Utilities` est quasi-constante (1457/1458 valent `AllPub`) et donc retirée.

La cardinalité de chaque colonne nominale est **vérifiée directement sur `X_train`** (et non figée dans une liste) : le seuil de 6 modalités tranche automatiquement entre les deux familles d'encodage.

In [8]:
num_cols = X_train.select_dtypes(include=np.number).columns.tolist()

num_zero_cols = [
    'GarageYrBlt', 'GarageArea', 'GarageCars', 'BsmtFinSF1',
    'BsmtFinSF2', 'BsmtUnfSF', 'TotalBsmtSF', 'BsmtFullBath',
    'BsmtHalfBath', 'MasVnrArea',
]
# Certaines colonnes zero-fill ont été retirées (colinéarité) — on filtre pour
# n'exposer au ColumnTransformer que les colonnes réellement présentes.
num_zero_cols = [c for c in num_zero_cols if c in X_train.columns]
num_median_cols = [c for c in num_cols if c not in num_zero_cols]

ordinal_cols = [
    'ExterQual', 'ExterCond', 'BsmtQual', 'BsmtCond', 'HeatingQC',
    'KitchenQual', 'FireplaceQu', 'GarageQual', 'GarageCond', 'PoolQC',
]
ordinal_cats = [['None', 'Po', 'Fa', 'TA', 'Gd', 'Ex']] * len(ordinal_cols)

all_cat_cols = X_train.select_dtypes(include=['object', 'string']).columns.tolist()

# Functional : branche dédiée (imputation 'Typ' + TargetEncoder).
functional_col = ['Functional'] if 'Functional' in all_cat_cols else []

# Nominales = tout le catégoriel hors ordinal, hors Functional, hors Utilities (quasi-constante).
nominal_cols = [c for c in all_cat_cols
                if c not in ordinal_cols and c not in functional_col and c != 'Utilities']

# Séparation haute / basse cardinalité, calculée sur X_train (seuil = 6 modalités).
CARD_THRESHOLD = 6
card = X_train[nominal_cols].nunique().sort_values(ascending=False)
nominal_highcard_cols = card[card >= CARD_THRESHOLD].index.tolist()
nominal_lowcard_cols = card[card < CARD_THRESHOLD].index.tolist()

# Colonnes où NA = absence réelle ('None'), à imputer par une constante plutôt que par le mode
# (ex. BsmtFinType1 vaut 'None' quand il n'y a pas de sous-sol).
nominal_none_cols = [
    'Alley', 'BsmtFinType1', 'BsmtFinType2', 'BsmtExposure',
    'GarageType', 'GarageFinish', 'Fence', 'MiscFeature', 'MasVnrType',
]
nominal_lowcard_none = [c for c in nominal_lowcard_cols if c in nominal_none_cols]
nominal_lowcard_mode = [c for c in nominal_lowcard_cols if c not in nominal_none_cols]
nominal_highcard_none = [c for c in nominal_highcard_cols if c in nominal_none_cols]
nominal_highcard_mode = [c for c in nominal_highcard_cols if c not in nominal_none_cols]

# --- Audit de cardinalité (documente le choix d'encodage) ---
print("Audit de cardinalité des colonnes nominales (sur X_train) :")
print(f"{'Colonne':16s} {'modalités':>10s}   bucket")
for col, n in card.items():
    enc = 'TargetEncoder (haute)' if n >= CARD_THRESHOLD else 'OneHot (basse)'
    imp = "'None'" if col in nominal_none_cols else 'mode'
    print(f"  {col:16s} {n:>8d}     {enc:22s} [imput. {imp}]")
print(f"  {'Functional':16s} {X_train['Functional'].nunique():>8d}     TargetEncoder (branche 'Typ')")
print()
print(f"  num_zero          ({len(num_zero_cols):2d} cols)")
print(f"  num_median        ({len(num_median_cols):2d} cols, dont LotFrontage par quartier)")
print(f"  ordinal           ({len(ordinal_cols):2d} cols)")
print(f"  nominal_highcard  ({len(nominal_highcard_cols):2d} cols  -> TargetEncoder)  [none={len(nominal_highcard_none)}, mode={len(nominal_highcard_mode)}]")
print(f"  nominal_lowcard   ({len(nominal_lowcard_cols):2d} cols  -> OneHot)        [none={len(nominal_lowcard_none)}, mode={len(nominal_lowcard_mode)}]")
print(f"  functional        ({len(functional_col):2d} col   -> TargetEncoder)")

Audit de cardinalité des colonnes nominales (sur X_train) :
Colonne           modalités   bucket
  Neighborhood           25     TargetEncoder (haute)  [imput. mode]
  Exterior2nd            16     TargetEncoder (haute)  [imput. mode]
  Exterior1st            15     TargetEncoder (haute)  [imput. mode]
  MSSubClass             15     TargetEncoder (haute)  [imput. mode]
  SaleType                9     TargetEncoder (haute)  [imput. mode]
  Condition1              9     TargetEncoder (haute)  [imput. mode]
  Condition2              8     TargetEncoder (haute)  [imput. mode]
  HouseStyle              8     TargetEncoder (haute)  [imput. mode]
  BsmtFinType1            6     TargetEncoder (haute)  [imput. 'None']
  GarageType              6     TargetEncoder (haute)  [imput. 'None']
  SaleCondition           6     TargetEncoder (haute)  [imput. mode]
  Heating                 6     TargetEncoder (haute)  [imput. mode]
  Foundation              6     TargetEncoder (haute)  [imput. mode]
  

### 3.6.2 `preprocessor_scaled` — pour les modèles sensibles à l'échelle

**Consommé par :** OLS, Ridge, Lasso, ElasticNet, KNN, SVR, MLPRegressor.

**Pourquoi la mise à l'échelle :** ces modèles reposent soit sur une *distance euclidienne* (KNN, noyau SVR), soit sur une *pénalité de norme* (Ridge/Lasso/ElasticNet — la régularisation suppose que toutes les variables sont sur une échelle comparable, sinon les grandes valeurs absorbent toute la pénalité), soit sur une *descente de gradient* (MLP). OLS sans pénalité n'a pas strictement besoin du scaler mais on l'inclut pour stabiliser la condition numérique.

**Pourquoi `TargetEncoder` plutôt que `OneHotEncoder` (directive du prof) :** encoder en *one-hot* toutes les nominales fait exploser l'espace de features (`Neighborhood` à 25 modalités, `Exterior1st/2nd`, `MSSubClass`… → 250+ colonnes creuses dans la version naïve). Conséquences : malédiction de la dimension (catastrophique pour KNN et le noyau RBF), coefficients instables sur les modalités rares, et perte de toute notion de proximité entre modalités. On utilise donc `TargetEncoder(cv=5, smooth='auto')` pour les nominales à **haute cardinalité** : chaque modalité est remplacée par la moyenne de la cible (`y_log`) qui lui est associée, estimée par **validation croisée interne** (pas de fuite : l'encodage d'une ligne n'utilise jamais sa propre cible) et **lissée vers la moyenne globale** pour les modalités rares. On garde le one-hot uniquement pour les nominales à **basse cardinalité** (≤ 5 modalités), où il reste bon marché et lisible.

Comme la sortie du `TargetEncoder` vit sur l'échelle de `y_log` (≈ 11–13), elle est **elle aussi standardisée** dans la variante `scaled`, sinon elle dominerait la distance / la pénalité.

**`LotFrontage` par quartier :** plutôt qu'une médiane globale, on impute `LotFrontage` par la **médiane de son quartier** (`Neighborhood`), ajustée sur `X_train` uniquement (transformateur `NeighborhoodMedianImputer` placé en tête du pipeline, car il a besoin des deux colonnes brutes simultanément).

In [9]:
class NeighborhoodMedianImputer(BaseEstimator, TransformerMixin):
    """Impute LotFrontage par la médiane de son quartier (ajustée sur le train).

    Doit précéder le ColumnTransformer car il a besoin de `Neighborhood` ET
    `LotFrontage` simultanément. Pass-through sur la structure de colonnes :
    `get_feature_names_out` renvoie les noms d'entrée pour rester compatible
    avec l'introspection des features en aval (extraction des coefficients OLS).
    """

    def __init__(self, group_col='Neighborhood', target_col='LotFrontage'):
        self.group_col = group_col
        self.target_col = target_col

    def fit(self, X, y=None):
        self.feature_names_in_ = np.asarray(X.columns)
        self.n_features_in_ = X.shape[1]
        if self.target_col in X.columns and self.group_col in X.columns:
            self.medians_ = X.groupby(self.group_col)[self.target_col].median()
            self.global_median_ = X[self.target_col].median()
        else:
            self.medians_, self.global_median_ = None, None
        return self

    def transform(self, X):
        X = X.copy()
        if self.medians_ is None:
            return X
        mask = X[self.target_col].isna()
        X.loc[mask, self.target_col] = (
            X.loc[mask, self.group_col].map(self.medians_).fillna(self.global_median_)
        )
        return X

    def get_feature_names_out(self, input_features=None):
        if input_features is None:
            return self.feature_names_in_
        return np.asarray(input_features)


def _build_column_transformer(scale: bool) -> ColumnTransformer:
    def _num_steps(imputer):
        steps = [('imputer', imputer)]
        if scale:
            steps.append(('scaler', StandardScaler()))
        return steps

    def _target_steps(imputer):
        # Imputation -> TargetEncoder (cross-fit, anti-fuite) -> StandardScaler (variante scaled).
        steps = [
            ('imputer', imputer),
            ('encoder', TargetEncoder(target_type='continuous', smooth='auto', cv=5, random_state=RANDOM_STATE)),
        ]
        if scale:
            steps.append(('scaler', StandardScaler()))
        return steps

    impute_zero = SimpleImputer(strategy='constant', fill_value=0)
    impute_median = SimpleImputer(strategy='median')
    impute_none = SimpleImputer(strategy='constant', fill_value='None')
    impute_mode = SimpleImputer(strategy='most_frequent')
    impute_typ = SimpleImputer(strategy='constant', fill_value='Typ')

    return ColumnTransformer(
        transformers=[
            ('num_zero',     Pipeline(_num_steps(impute_zero)),     num_zero_cols),
            ('num_median',   Pipeline(_num_steps(impute_median)),   num_median_cols),
            ('ord',          Pipeline([
                                ('imputer', impute_none),
                                ('encoder', OrdinalEncoder(categories=ordinal_cats, handle_unknown='use_encoded_value', unknown_value=-1)),
                             ]), ordinal_cols),
            ('nom_low_none', Pipeline([
                                ('imputer', impute_none),
                                ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
                             ]), nominal_lowcard_none),
            ('nom_low_mode', Pipeline([
                                ('imputer', impute_mode),
                                ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
                             ]), nominal_lowcard_mode),
            ('nom_high_none', Pipeline(_target_steps(impute_none)), nominal_highcard_none),
            ('nom_high_mode', Pipeline(_target_steps(impute_mode)), nominal_highcard_mode),
            ('functional',    Pipeline(_target_steps(impute_typ)),  functional_col),
        ],
        remainder='drop',
    )


def _with_lotfrontage(ct: ColumnTransformer) -> Pipeline:
    """Préfixe le ColumnTransformer par l'imputation LotFrontage par quartier."""
    return Pipeline([('lotfront', NeighborhoodMedianImputer()), ('ct', ct)])


preprocessor_scaled = _with_lotfrontage(_build_column_transformer(scale=True))

### 3.6.3 `preprocessor_encoded` — pour les arbres sklearn

**Consommé par :** DecisionTreeRegressor, RandomForestRegressor, GradientBoostingRegressor, AdaBoostRegressor.

**Pourquoi :** un arbre de décision **partitionne l'espace par seuils** (ex. `GrLivArea > 1500`). Cette opération est **invariante par toute transformation monotone croissante** : standardiser une variable ne change ni l'ordre des observations ni les seuils possibles. Le `StandardScaler` est donc une opération *inutile* (sans dégrader la performance). On garde la même structure d'imputation et d'encodage que `preprocessor_scaled` (y compris `TargetEncoder` pour les nominales à haute cardinalité — qui réduit aussi le nombre de seuils candidats par rapport au one-hot), mais sans mise à l'échelle.

In [10]:
preprocessor_encoded = _with_lotfrontage(_build_column_transformer(scale=False))

### 3.6.4 `preprocessor_native` — pour les boosters modernes

**Consommé par :** XGBoost, LightGBM, CatBoost.

**Pourquoi :** ces bibliothèques implémentent leurs propres mécanismes pour les valeurs manquantes (`NaN`-aware splits) et pour les variables catégorielles (`enable_categorical=True` pour XGBoost ≥ 1.6, `categorical_feature` pour LightGBM, `cat_features` pour CatBoost). Imputer ou encoder en amont **gaspille de l'information** : le modèle perd la capacité de traiter NaN comme un signal en soi, et l'encodage explicite fait perdre la gestion native. Le préprocesseur natif se contente donc de convertir les colonnes `object` en `pd.Categorical` en mémorisant les catégories observées à l'entraînement, ce qui garantit la cohérence train/test. `MSSubClass` (forcé en chaîne en 3.4) y est désormais traité comme une vraie catégorielle native, et non comme un entier.

**Cas particulier CatBoost.** CatBoost ne gère pas les `NaN` dans les colonnes catégorielles (contrairement à XGBoost et LightGBM). On fournit donc, pour ce modèle uniquement, un prétraitement dédié `CatBoostPrep` qui comble ces `NaN` par une catégorie `'NAN'` (les numériques restent intacts). Ce prétraitement n'affecte que le pipeline CatBoost.

In [11]:
class NativeCategoricalTransformer(BaseEstimator, TransformerMixin):
    """Coerce object/string columns to pd.Categorical with train-fixed categories.

    Numerical columns and NaN cells pass through untouched — the consumer
    model (XGBoost/LightGBM/CatBoost) handles them natively. Test-set values
    not seen at fit time are coerced to NaN (consumer model handles them too).
    """

    def fit(self, X, y=None):
        self.cat_cols_ = X.select_dtypes(include=['object', 'string']).columns.tolist()
        self.categories_ = {
            col: X[col].astype('category').cat.categories
            for col in self.cat_cols_
        }
        return self

    def transform(self, X):
        X = X.copy()
        for col in self.cat_cols_:
            known = self.categories_[col]
            # Coerce unknown values to NaN before building the Categorical,
            # to avoid pandas4 deprecation warning + match consumer expectation.
            col_vals = X[col].where(X[col].isin(known))
            X[col] = pd.Categorical(col_vals, categories=known)
        return X


preprocessor_native = NativeCategoricalTransformer()


In [12]:
class CatBoostPrep(BaseEstimator, TransformerMixin):
    """Prétraitement minimal pour CatBoost (gestion native des catégorielles).

    Contrairement à XGBoost et LightGBM, CatBoost ne gère pas les NaN dans les
    colonnes catégorielles (issue #571) : on les comble par une catégorie
    dédiée 'NAN'. Les colonnes numériques sont laissées intactes (CatBoost gère
    leurs NaN nativement). Transformation sur copie — aucune mutation en amont.
    """

    def fit(self, X, y=None):
        self.cat_cols_ = X.select_dtypes(include=['object', 'string']).columns.tolist()
        return self

    def transform(self, X):
        X = X.copy()
        for col in self.cat_cols_:
            X[col] = X[col].astype('object').where(X[col].notna(), 'NAN').astype(str)
        return X


class CatBoostRegressorCV(BaseEstimator, RegressorMixin):
    """Wrapper sklearn-compatible autour de CatBoostRegressor.

    `CatBoostRegressor(cat_features=...)` ne passe pas `sklearn.clone` (son
    `get_params` ne restitue pas l'objet `cat_features` à l'identique), ce qui
    casse `cross_val_score` / `StackingRegressor`. Ce wrapper stocke les
    hyperparamètres comme attributs simples (clone OK) et construit le modèle
    au `fit`. `cat_features` = liste des noms de colonnes catégorielles.
    """

    def __init__(self, cat_features=None, iterations=500, learning_rate=0.05,
                 depth=6, random_seed=RANDOM_STATE):
        self.cat_features = cat_features
        self.iterations = iterations
        self.learning_rate = learning_rate
        self.depth = depth
        self.random_seed = random_seed

    def fit(self, X, y=None):
        from catboost import CatBoostRegressor
        self.model_ = CatBoostRegressor(
            iterations=self.iterations, learning_rate=self.learning_rate,
            depth=self.depth, cat_features=self.cat_features,
            random_seed=self.random_seed, verbose=False,
        )
        self.model_.fit(X, y)
        return self

    def predict(self, X):
        return self.model_.predict(X)

## 3.7 Sentinelle anti-fuite

Le préprocesseur est désormais la *source unique de vérité* pour le nettoyage. Pour empêcher qu'un futur contributeur ne réintroduise un nettoyage en amont (e.g. un `fillna()` global), on ajoute une assertion explicite : `X_train` **doit** contenir des NaN à ce stade. Si quelqu'un impute en amont, l'assertion casse et la dérive est rattrapée.

In [13]:
assert X_train.isna().sum().sum() > 0, (
    "Sentinelle anti-fuite — le pipeline doit recevoir des données brutes avec des NaN à imputer. "
    "Si cette assertion casse, un nettoyage en amont a été réintroduit."
)

## 3.8 Fonctions utilitaires partagées

Ces fonctions sont importées (via `%run`) par tous les notebooks de modélisation et par le notebook d'évaluation. Elles définissent :

- `rmsle_score(y_true_log, y_pred_log)` — la métrique du projet (RMSE en espace log = RMSLE en espace brut).
- `cv_rmsle(pipe, X, y_log, cv=5)` — RMSLE par cross-validation 5-folds.
- `predicted_vs_actual_plot(...)` — le scatter exigé par la consigne (« chaque modèle doit avoir un scatter predicted-vs-actual »).
- `publish_result(...)` — sérialise les résultats d'un modèle dans `results/family_<name>.json` (lu par `4_evaluation.ipynb`).

In [14]:
def rmsle_score(y_true_log, y_pred_log) -> float:
    """RMSE sur la cible log-transformée = RMSLE sur les prix bruts."""
    return float(np.sqrt(mean_squared_error(y_true_log, y_pred_log)))


def cv_rmsle(pipe, X, y_log, cv: int = 5) -> float:
    """RMSLE moyenne en cross-validation k-fold (scoring scikit-learn `neg_root_mean_squared_error`)."""
    scores = cross_val_score(
        pipe, X, y_log, cv=cv,
        scoring='neg_root_mean_squared_error',
        n_jobs=-1,
    )
    return float(-scores.mean())


def predicted_vs_actual_plot(y_true_log, y_pred_log, title: str = 'Predicted vs Actual', ax=None):
    """Scatter predicted-vs-actual avec diagonale parfaite en pointillé rouge."""
    if ax is None:
        _, ax = plt.subplots(figsize=(6, 6))
    ax.scatter(y_true_log, y_pred_log, alpha=0.4, s=20)
    lo = float(min(np.min(y_true_log), np.min(y_pred_log)))
    hi = float(max(np.max(y_true_log), np.max(y_pred_log)))
    ax.plot([lo, hi], [lo, hi], 'r--', lw=1.5, label='y = x')
    ax.set_xlabel('Prix réel (log)')
    ax.set_ylabel('Prix prédit (log)')
    ax.set_title(title)
    ax.legend(loc='upper left')
    return ax


RESULTS_DIR = Path('results')
RESULTS_DIR.mkdir(exist_ok=True)


def publish_result(family: str, model: str,
                   cv_rmsle: float = None,
                   holdout_rmsle: float = None,
                   fit_time_s: float = None,
                   params: dict = None,
                   notes: str = None) -> Path:
    """Append/upsert (par `model` name) un résultat dans results/family_<family>.json."""
    path = RESULTS_DIR / f'family_{family}.json'
    data = json.loads(path.read_text()) if path.exists() else []
    data = [r for r in data if r.get('model') != model]
    data.append({
        'family': family,
        'model': model,
        'cv_rmsle': cv_rmsle,
        'holdout_rmsle': holdout_rmsle,
        'fit_time_s': fit_time_s,
        'params': params,
        'notes': notes,
    })
    path.write_text(json.dumps(data, indent=2, default=str))
    return path

## 3.9 Vérification finale

Vérification rapide que les trois préprocesseurs et tous les helpers sont définis. **On ne `fit()` rien ici** — chaque notebook de modélisation `fit`tera son propre `Pipeline(preprocessor_X, model)` une seule fois.

In [15]:
print("Variantes de préprocesseur :")
print(f"  preprocessor_scaled  : {type(preprocessor_scaled).__name__} (lotfront + ColumnTransformer, avec StandardScaler)")
print(f"  preprocessor_encoded : {type(preprocessor_encoded).__name__} (lotfront + ColumnTransformer, sans scaler)")
print(f"  preprocessor_native  : {type(preprocessor_native).__name__}")
print()
print("Helpers : rmsle_score, cv_rmsle, predicted_vs_actual_plot, publish_result")
print()
print(f"Données prêtes — X_train {X_train.shape}, X_test {X_test.shape}")

Variantes de préprocesseur :
  preprocessor_scaled  : Pipeline (lotfront + ColumnTransformer, avec StandardScaler)
  preprocessor_encoded : Pipeline (lotfront + ColumnTransformer, sans scaler)
  preprocessor_native  : NativeCategoricalTransformer

Helpers : rmsle_score, cv_rmsle, predicted_vs_actual_plot, publish_result

Données prêtes — X_train (1166, 83), X_test (292, 83)
